# Upoznavanje sa kreiranim skupom podataka

In [8]:
import argparse
import re
import statistics
from dataclasses import dataclass, field, asdict
from pathlib import Path
import pandas as pd
from collections import Counter
import langdetect
import matplotlib
import tqdm

## Analiza tekstualnih dokumenata iz Project Gutenberg skupa podataka

Skripta koja se koristi za analizu tekstualnih dokumenata iz Project Gutenberg skupa podataka.

**Cilj**: Eksplorativna analiza podataka i upoznavanje sa knjiga kroz ceo skup.

**Pitanja na koje analiza odgovara**:

1. START/END marker provera - Koliko knjiga sadrži nekakvu vrstu markera koji označavaju početak/kraj knjige? Radiće se upotrebom regularnih izraza. Korisno kako bi se filtriranje uradilo uz izbegavanje metapodataka knjiga.

1. Proveriti koliko ukupno i koliko često knjige imaju [] i provera da li je taj tekst koristan ili ga je moguće odbaciti - zapisati kao multiset (tekst: broj pojavljivanja).

1. Pogledati koliki broj knjiga nema korisnu sadržinu nakon izdvajanja markera i odbacivanja teksta u [], što se procenjuje na 2 načina (dovoljno je 1 da važi): tekst je manji od 100 reči ili odbačeno je 50% teksta tokom ranijeg procesa. Za same knjige će biti navedeno koliko je reči bilo pre, a koliko posle svega. Takođe će postojati i jedno bool polje for_deleting - true vrednost označava da se knjiga neće unositi u baze podataka, odnosno false znači da će se unositi.

1. Provera postojanja poglavlja - Koliko knjiga uopšte ima poglavlja u svojoj sadržini? Korisno za potrebe structure aware chunking dela strategije.

1. Distribucija dužine poglavlja kao i broja samih poglavlja i celih knjiga koje nemaju poglavlja. Radi se paralelni prikaz dužina knjiga sa i bez poglavlja. Računaće se: min/max/mean/median, prikazaće se histogram i boxplot. Pored ovoga, za svaku knjigu će se navesti min/max/mean/median po poglavljima, ali isto tako i polje koje će nam biti značajno kasnije: has_chapters - korisno za samu pretragu jer se time naglašava da knjiga nema uopšte poglavlja i takođe se preskače pretraga po chapter_summary kolekciji.

1. Provera da li knjige imaju neki vid sadržaja dela na početku ili kraju obrađene knjige (10% knjige) - može biti korisno za potrebe structure aware chunking dela strategije.

1. Heuristika min/max/mean/median za svaku knjigu (broj reči po liniji, gustina interpunkcije po liniji) - Korisno kako bi se uradila gruba klasifikacija na prozu/poeziju.

1. Statistika samog pojavljivanja jezika u knjigama. Za potrebe ovoga koristiće se neka biblioteka kao što je langdetect. Pokazaće nam koliku važnost igra multilingualnost samih modela: LLM za sažetke i modela ugrađivanja.


**Rezultat**: Nakon toga je potrebno kreirati jedan novi fajl u kom će se navesti odgovori na ranija pitanja, što može biti korisno za kasnije procedure chunking-a, unosa, kreiranja sažetaka itd.   

In [9]:
# NAPOMENE: 
# 1. Nije dobro kreiran sam regex pa samim tim ni statistika koja ide uz njega
# 2. Deluje da je provera za sadrzaj dobra
# 3. Zaista je proveravano da je u pitanju stvarno poglavlje dakle CHAPTER (ideja je da bi trebalo nesto po sadrzaju i ovo kombinovati)
# 4. Statistika pokazuje na neki cudne outliers
# 5. Deluje u redu.

# ---------------------------------------------------------------------------
# Regex-i
# ---------------------------------------------------------------------------

# Generički pattern za regex-e koji koristimo da bismo uhvatili što je moguće više knjiga, jer nam nisu svi rasporedi početaka poznati.
GENERIC_START_RE = re.compile(r"\*\*\*\s*START.{0,300}?\*\*\*", re.IGNORECASE | re.DOTALL)
GENERIC_END_RE = re.compile(r"\*\*\*\s*END.{0,300}?\*\*\*", re.IGNORECASE | re.DOTALL)

# Sadržaj (Table of contents) - tražimo reč "content" na svojoj liniji (case-insensitive). Služi za proveru da li je TOC u prvih 10% ili poslednjih 10% knjige.
TOC_HEADING_RE = re.compile(r"^\s*(contents|table of contents|content|table of content)\s*$", re.IGNORECASE | re.MULTILINE)

# Chapter marker - (npr. "Chapter 1", "CHAPTER I.", "Chapter 1: The Beginning"
CHAPTER_RE = re.compile(r"^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b\.?\s*(?:[:\-–—]\s*\S.*)?\s*$", re.IGNORECASE | re.MULTILINE)

# Tekst u uglastim zagradama, u pitanju su obično nekakve napomene.
BRACKET_TEXT_RE = re.compile(r"\[([^\[\]]*)\]", re.DOTALL)

# Regex-i za reči i kraj rečenice
WORD_RE = re.compile(r"\S+")
SENTENCE_END_RE = re.compile(r"[.!?]")

# ---------------------------------------------------------------------------
# Korisne konstante
# ---------------------------------------------------------------------------
WORDS_PER_LINE_POETRY = 8
MIN_WORDS_AFTER_CLEANUP = 100
REMOVAL_THRESHOLD = 0.5
MIN_FOR_MULTIPLE_ELEMENTS = 2

# OVA 2 SE MORAJU PODESITI U SKLADU SA HISTOGRAMOM
SHORT_LINE_RATIO_THRESHOLD = 0.35
MEDIAN_WORDS_PER_LINE_THRESHOLD = 10

In [10]:
@dataclass
class BookProbeResult:
    """Rezultat analize za jednu knjigu - jedan red u finalnom DataFrame-u"""

    # Generalno o fajlu i knjizi
    file_name: str
    file_size_bytes: int = 0

    # (1) START/END markeri - Dovoljna su ova 2 iz razloga što tražimo samo generički početak, ne znamo šta se sve krije među knjigama.
    has_generic_start: bool = False
    has_generic_end: bool = False
    total_word_count: int = 0 # posle čišćenja markera (ako je to moguće) računamo koliko ima reči inače je ukupan broj reči

    # (2) Tekst u []
    bracket_text_count: int = 0
    bracket_texts: list = field(default_factory=list)
    words_removed_from_brackets: int = 0 # koliko reči je obrisano 

    # (3) Korisna sadržina knjige posle čišćenja (koraci 1 i 2)
    word_count_before_cleanup: int = 0 # = total_word_count
    word_count_after_cleanup: int = 0  # posle uklanjanja [...] teksta
    fraction_removed_by_cleanup: float = 0.0
    too_short_after_cleanup: bool = False  # < 100 reči posle čišćenja
    over_half_removed: bool = False  # odbačeno >= 50% teksta
    for_deleting: bool = True  # True = ne unositi u bazu (bilo koji od gornja 2 uslova)

    # (4) Postojanje poglavlja
    chapter_marker_count: int = 0
    has_chapters: bool = False # potrebno je da postoje najmanje 2 oznake za poglavlja
    
    # (5) Dužina poglavlja / cele knjige
    chapter_word_counts: list = field(default_factory=list)  # prazna lista ako nema poglavlja, inače ima dužinu poglavlja

    # (6) Sadržaj (TOC)
    has_toc_heading: bool = False
    toc_position_ratio: float = -1.0  # 0.0 = na početku, 1.0 = na kraju, -1 = nema
    has_toc_in_first_10pct: bool = False
    has_toc_in_last_10pct: bool = False

    # (7) Proza/poezija heuristika
    avg_words_per_line: float = 0.0
    median_words_per_line: float = 0.0
    punctuation_density: float = 0.0  # broj interpunkcijskih znakova / broj karaktera
    short_line_ratio: float = 0.0  # % linija kraćih od 8 reči (relativno slab indikator stihova)
    likely_poetry: bool = False  # gruba oznaka za poeziju

    # (8) Detekcija jezika
    detected_language: str = "unknown"  
    detected_language_confidence: float = 0.0
    is_multilingual_candidate: bool = False  # ako postoji barem 2 jezika sa smislenom verovatnoćom u istoj knjizi
    
    # Greške/napomene - da ništa ne prođe nezapaženo
    notes: str = ""

In [11]:
def probe_single_ebook(file_path: Path) -> BookProbeResult:
    """Služi za analizu jedne knjige na prosleđenoj putanji, a greške se zapisuju u polje notes."""

    result = BookProbeResult(file_name=file_path.name)
    notes = []

    try:
        result.file_size_bytes = file_path.stat().st_size
        raw_text = file_path.read_text(encoding="utf-8", errors="replace")
    except Exception as e:
        result.notes = f"READ_ERROR: {e}"
        return result

    if not raw_text.strip():
        result.notes = "EMPTY_FILE"
        return result

    # (1) --- START/END marker provera ---
    start_match = GENERIC_START_RE.search(raw_text)
    end_match = GENERIC_END_RE.search(raw_text)
    result.has_generic_start = start_match is not None
    result.has_generic_end = end_match is not None

    # Izdvajanje tela knjige između markera, ako je to moguće
    if start_match and end_match:
        body = raw_text[start_match.end():end_match.start()]
    else:
        body = raw_text
        notes.append("NO_CLEAN_BODY_EXTRACTION_used_full_text")

    body = body.strip()
    if not body:
        result.notes = "; ".join(notes + ["EMPTY_BODY_AFTER_MARKER_STRIP"])
        return result

    # Otkrivamo dužinu teksta posle čišćenja markera
    all_words = WORD_RE.findall(body)
    result.total_word_count = len(all_words)

    # (2) Tekst u []
    bracket_matches = list(BRACKET_TEXT_RE.finditer(body))
    result.bracket_text_count = len(bracket_matches)
    result.bracket_texts = [m.group(1).strip() for m in bracket_matches]
    
    # (3) Korisna sadržina knjige posle čišćenja (koraci 1 i 2)
    cleaned_body = BRACKET_TEXT_RE.sub(" ", body)
    words_before = len(all_words)
    words_after = len(WORD_RE.findall(cleaned_body))

    result.word_count_before_cleanup = words_before
    result.word_count_after_cleanup = words_after

    result.words_removed_from_brackets = max(words_before - words_after, 0)
    result.fraction_removed_by_cleanup = (round((words_before - words_after) / words_before, 4) if words_before > 0 else 0.0)

    result.too_short_after_cleanup = words_after < MIN_WORDS_AFTER_CLEANUP
    result.over_half_removed = result.fraction_removed_by_cleanup >= 0.5
    result.for_deleting = result.too_short_after_cleanup or result.over_half_removed

    # --- (4) Provera postojanja poglavlja ---
    chapter_matches = list(CHAPTER_RE.finditer(body))
    result.chapter_marker_count = len(chapter_matches)
    result.has_chapters = len(chapter_matches) >= MIN_FOR_MULTIPLE_ELEMENTS

    # --- (5) Distribucija dužine poglavlja (ako postoje) ---
    if result.has_chapters:
        positions = [m.start() for m in chapter_matches] + [len(body)]
        chapter_word_counts = []
        for start_pos, end_pos in zip(positions[:-1], positions[1:]):
            segment = body[start_pos:end_pos]
            chapter_word_counts.append(len(WORD_RE.findall(segment)))
        result.chapter_word_counts = chapter_word_counts

    # --- (6) Sadržaj (TOC) provera ---
    toc_match = TOC_HEADING_RE.search(body)
    result.has_toc_heading = toc_match is not None
    
    body_len = max(len(body), 1)
    ten_pct = body_len * 0.1

    if toc_match:
        result.toc_position_ratio = round(toc_match.start() / max(len(body), 1), 4)
        if toc_match.start() <= ten_pct:
            result.has_toc_in_first_10pct = True
        elif toc_match.start() >= body_len - ten_pct:
            result.has_toc_in_last_10pct = True

    # --- (7) Proza/poezija gruba klasifikacija ---
    punctuation_count = sum(1 for ch in body if ch in ".,;:!?")
    result.punctuation_density = round(punctuation_count / max(len(body), 1), 4)

    lines = [ln.strip() for ln in body.splitlines() if ln.strip()]
    if lines:
        words_per_line = [len(WORD_RE.findall(ln)) for ln in lines]
        result.avg_words_per_line = round(statistics.mean(words_per_line), 4)
        result.median_words_per_line = round(statistics.median(words_per_line), 4)
        short_lines = sum(1 for w in words_per_line if w < WORDS_PER_LINE_POETRY)
        result.short_line_ratio = round(short_lines / len(lines), 4)
        result.likely_poetry = (result.median_words_per_line <= MEDIAN_WORDS_PER_LINE_THRESHOLD or result.short_line_ratio >= SHORT_LINE_RATIO_THRESHOLD)

    # (8) Detekcija jezika
    lang_sample = cleaned_body.strip()[:20000]
    if lang_sample:
        try:
            lang_probs = langdetect.detect_langs(lang_sample)
            if lang_probs:
                result.detected_language = lang_probs[0].lang
                result.detected_language_confidence = round(lang_probs[0].prob, 4)

                # Označavamo jezik da je kandidat za multilingualnu detekciju (čisto informativno)
                if len(lang_probs) > 1 and lang_probs[1].prob >= 0.25:
                    result.is_multilingual_candidate = True
        except langdetect.LangDetectException:
            result.detected_language = "error"
            notes.append("LANGDETECT_FAILED")
    else:
        result.detected_language = "unknown"

    # Vraćanje informacija o samoj knjizi korisniku
    result.notes = "; ".join(notes)
    return result

In [12]:
def probe_ebooks(input_dir: Path) -> pd.DataFrame:
    """Prolazi kroz sve fajlove koji se nalaze u input_dir i vraća DataFrame sa jednim redom po knjizi."""

    ebook_files = sorted(input_dir.glob("*"))
    if not ebook_files:
        raise FileNotFoundError(f"Ne postoje fajlovi u {input_dir}!")

    results = [probe_single_ebook(file_path) for file_path in tqdm.tqdm(ebook_files, desc="Analiza knjiga")]
    df = pd.DataFrame([asdict(r) for r in results])
    return df

In [13]:
def summarize_ebooks_stats(df: pd.DataFrame) -> str:
    n = len(df)
    lines = []
    lines.append(f"Ukupno analizirano knjiga: {n}\n")

    # (1) START/END markeri
    lines.append("-" * 70)
    lines.append("1) START/END markeri")
    lines.append("-" * 70)

    both_existing = ((df["has_generic_start"]) & (df["has_generic_end"])).sum()
    lines.append(f"  Postojanje START markera:                          {df['has_generic_start'].sum()} / {n} ({100*df['has_generic_start'].sum()/n:.1f}%)")
    lines.append(f"  Postojanje END markera:                            {df['has_generic_end'].sum()} / {n} ({100*df['has_generic_end'].sum()/n:.1f}%)")
    lines.append(f"  Oba markera prisutna (čisto izdvajanje moguće):    {both_existing} / {n} ({100*both_existing/n:.1f}%)")

    none_existing = ((~df["has_generic_start"]) & (~df["has_generic_end"])).sum()
    lines.append(f"  Nema START marker:                                 {(~df['has_generic_start']).sum()}")
    lines.append(f"  Nema END marker:                                   {(~df['has_generic_end']).sum()}")
    lines.append(f"  Nijedan marker nije prisutan:                      {none_existing} / {n} ({100*none_existing/n:.1f}%)")

    if ((~df["has_generic_start"]) | (~df["has_generic_end"])).sum() > 0:
        problem_files = df[(~df["has_generic_start"]) | (~df["has_generic_end"])]['file_name'].tolist()
        lines.append(f"  >> Fajlovi bez ikakvih markera: {problem_files}")
    
    # (4) Statistika o poglavljima
    lines.append("")
    lines.append("-" * 70)
    lines.append("4) Pokrivenost poglavljima")
    lines.append("-" * 70)
    has_chapters = df["has_chapters"].sum()
    lines.append(f"  Knjige sa barem 2 oznake poglavlja:                {has_chapters} / {n} ({100*has_chapters/n:.1f}%)")
    lines.append(f"  Knjige bez formalnih poglavlja:                    {n - has_chapters} / {n} ({100*(n-has_chapters)/n:.1f}%)")
    
    # (5) Distribucija dužine knjiga i poglavlja (ako postoje)
    lines.append("")
    lines.append("-" * 70)
    lines.append("5) Distribucija dužine knjiga/poglavlja")
    lines.append("-" * 70)
    all_chapter_lengths = [wc for lst in df["chapter_word_counts"] for wc in lst]
    if all_chapter_lengths:
        lines.append(f"  Broj poglavlja ukupno (iz knjiga koje imaju poglavlja): {len(all_chapter_lengths)}")
        lines.append(f"  Dužina poglavlja po broju reči - min/median/mean/max: \n   {min(all_chapter_lengths)} / {statistics.median(all_chapter_lengths):.0f} / {statistics.mean(all_chapter_lengths):.0f} / {max(all_chapter_lengths)}")

    no_chapters_df = df[~df["has_chapters"]]
    if len(no_chapters_df) > 0:
        wc_no_chapters = no_chapters_df["total_word_count"]
        lines.append(f"  Za knjige bez poglavlja, dužina cele knjige (reči) - min/median/mean/max: \n   {wc_no_chapters.min()} / {wc_no_chapters.median():.0f} / {wc_no_chapters.mean():.0f} / {wc_no_chapters.max()}")
    
    # (6) Postojanje Sadržaja (TOC)
    lines.append("")
    lines.append("-" * 70)
    lines.append("6) Sadržaj (Table of Contents) na početku/kraju knjige")
    lines.append("-" * 70)
    toc_count = df["has_toc_heading"].sum()
    lines.append(f"  Ima prepoznatljiv 'Contents' heading:              {toc_count} / {n} ({100*toc_count/n:.1f}%)")
    if toc_count > 0:
        toc_positions = df[df["has_toc_heading"]]["toc_position_ratio"]
        near_start = (toc_positions <= 0.1).sum()
        near_end = (toc_positions >= 0.9).sum()
        lines.append(f"  Od toga, na početku knjige (pozicija <= 10%):      {near_start}")
        lines.append(f"  Od toga, na kraju knjige (pozicija >= 90%):        {near_end}")
        lines.append(f"  Od toga, u ostatku knjige:                         {toc_positions - near_end - near_start}")
    
    # (7) Proza/poezija statistika
    lines.append("")
    lines.append("-" * 70)
    lines.append("7) Gruba klasifikacija proza/poezija")
    lines.append("-" * 70)
    verse_count = df["likely_poetry"].sum()
    lines.append(f"  Knjige označene kao 'verovatno poezija':           {verse_count} / {n} ({100*verse_count/n:.1f}%)")
    lines.append(f"  Prosečan broj reči po liniji (ceo korpus):         {df['avg_words_per_line'].mean():.2f}")
    lines.append(f"  Prosečna medijana reči po liniji (ceo korpus):     {df['median_words_per_line'].mean():.2f}")
    lines.append(f"  Prosečna gustina interpunkcije (ceo korpus):       {df['punctuation_density'].mean():.2f}")

    return "\n".join(lines)

In [14]:
input_dir = Path("../data/book_texts/")
output_dir = Path("../data/stats/")
output_dir.mkdir(parents=True, exist_ok=True)

print("Početak analize knjiga iz Project Gutenberg skupa...")
df = probe_ebooks(input_dir)

parquet_path = output_dir / "ebooks_stats_results.parquet"
df.to_parquet(parquet_path, index=False)
print(f"Rezultati analize po knjigama su sačuvani u fajl: {parquet_path}")

# summary_text = summarize_ebooks_stats(df)
# summary_path = output_dir / "ebooks_summary.txt"
# summary_path.write_text(summary_text, encoding="utf-8")
# print(f"Analiza na nivou celog skupa podataka je u fajlu: {summary_path}")
# print("\n" + summary_text)

Početak analize knjiga iz Project Gutenberg skupa...


Analiza knjiga: 100%|██████████| 8174/8174 [14:03<00:00,  9.69it/s]


Rezultati analize po knjigama su sačuvani u fajl: ..\data\stats\ebooks_stats_results.parquet
